In [1]:
import matplotlib
import platform
print ("Operating system: ", platform.system())
if "Linux" in platform.system():
    %matplotlib tk
else:
    %matplotlib qt
    
#
import matplotlib.pyplot as plt
%autosave 180
%load_ext autoreload
%autoreload 2
import numpy as np

#
import scipy
import os
import time

#
import pickle 


#
from calibration.CalibrationTools import (CalibrationTools, get_binary_std_map, get_rois_stardist2d, get_img_std,
                                         save_calibration_data)
from drift.drift import (make_template, compute_drift_multi_frames, correct_drift, 
                         correct_drift_single_frame, template_generation, 
                         plot_mean_vs_template, make_motion_template_and_correct_data)
from utils.utils import smooth_ca_time_series, compute_dff0



Operating system:  Linux


Autosaving every 180 seconds


auto.py (22): IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html


In [2]:
#######################################################################
########### LOAD PRE-BMI DATA (e.g. 10-15mins recording) ##############
#######################################################################
fname = '/media/cat/4TBSSD/donato/DON-11733/data/Image_001_001.raw'
#fname  = r'F:\bmi\andres\20230112\DON2503\calibration\Image_001_001.raw'

# 
bmi_c = CalibrationTools(fname)

#
bmi_c.smooth_ca_time_series = smooth_ca_time_series
bmi_c.compute_dff0 = compute_dff0

#
bmi_c.subsample = 10 # for std computation downsample to every N'th frame; the more frames the better the rois;
                  #   TODO: use correlation instead?! might be much faster; it is quit fast in other implemenations

print ("DONE")

memmap :  (18000, 512, 512)
DONE


In [3]:
#####################################################
############### MOTION CORRECTION STEP ##############
#####################################################
# 
if False:
    start = time.time()
    bmi_c = make_motion_template_and_correct_data(bmi_c)
    plot_mean_vs_template(bmi_c)
    print ("total processing time: ", time.time()-start, " sec")
else:
    print ("Skipping template makgin step: ")
    bmi_c.template = np.mean(bmi_c.data[:1000],axis=0)
    
print ("DONE...")

Skipping template makgin step: 
DONE...


In [4]:
###################################################################
################# GET STD MAP TO GET CELL FOOTPRINTS ##############
###################################################################
# Filter data, then make map 
if True:
    std_map = bmi_c.filter_data_make_std_map()
    bmi_c.std_map = std_map
#
print ("...DONE...")


data into analysis:  (1800, 512, 512)
 gaussian filter width:  1 , order:  0
done filtering... (TO CHECK which axis are we filtering!!)
...DONE...


In [5]:
#################################################################
########### MAKE BINARY MAP FOR CELL REGISTRATOIN ###############
#################################################################

# get binary map
vmax = 1000
smin, smax = get_binary_std_map(bmi_c.std_map,vmax)


In [6]:
##################################################

bmi_c, img_std = get_img_std(smin, smax, bmi_c.std_map, bmi_c)


max proj values (vmin, vmax):  343.7499999999999 578.1249999999999


In [7]:
#########################################################
########### GENERATE CELL SEGMENTATION ##################
#########################################################
min_size_roi = 100  #<---- increase to exclude really small cells
max_size_roi = 800  #<----- decrease to exclude multi-cell artificats
bmi_c.roi_centres, bmi_c.footprints = get_rois_stardist2d(img_std,
                                                       min_size_roi,
                                                       max_size_roi)
print ("DONE...")

There are 4 registered models for 'StarDist2D':

Name                  Alias(es)
────                  ─────────
'2D_versatile_fluo'   'Versatile (fluorescent nuclei)'
'2D_versatile_he'     'Versatile (H&E nuclei)'
'2D_paper_dsb2018'    'DSB 2018 (from StarDist 2D paper)'
'2D_demo'             None
None
Found model '2D_versatile_fluo' for 'StarDist2D'.


2023-01-13 10:07:29.522340: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-13 10:07:29.530886: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-13 10:07:29.531150: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-13 10:07:29.534289: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from SysFS had negative value (-1), but there must be at least one NUMA node, so returning NUMA node zero
2023-01-13 10:07:29.534684: I tensorflow/stream_executor/cuda/cuda_gpu_executor.cc:975] successful NUMA node read from S

Loading network weights from 'weights_best.h5'.
Loading thresholds from 'thresholds.json'.
Using default values: prob_thresh=0.479071, nms_thresh=0.3.


2023-01-13 10:07:32.311009: I tensorflow/stream_executor/cuda/cuda_dnn.cc:384] Loaded cuDNN version 8400
2023-01-13 10:07:34.080361: W tensorflow/stream_executor/gpu/asm_compiler.cc:111] *** WARNING *** You are using ptxas 11.0.221, which is older than 11.1. ptxas before 11.1 is known to miscompile XLA code, leading to incorrect results or invalid-address errors.

You may not need to update to CUDA 11.1; cherry-picking the ptxas binary is often sufficient.
2023-01-13 10:07:34.082611: W tensorflow/stream_executor/gpu/asm_compiler.cc:230] Falling back to the CUDA driver for PTX compilation; ptxas does not support CC 8.6
2023-01-13 10:07:34.082651: W tensorflow/stream_executor/gpu/asm_compiler.cc:233] Used ptxas at ptxas
2023-01-13 10:07:34.082790: W tensorflow/stream_executor/gpu/redzone_allocator.cc:314] UNIMPLEMENTED: ptxas ptxas too old. Falling back to the driver to compile.
Relying on driver to perform ptx compilation. 
Modify $PATH to customize ptxas location.
This message will be 

1/1 [==============================] - 4s 4s/step


looping over cells: 100%|██████████████████████| 69/69 [00:00<00:00, 932.81it/s]


DONE...


In [8]:
#########################################################
########### REORDER CELLS BY SNR OR F0 ##################
#########################################################
order_type = 'snr'  # 'snr' or 'f0'
bmi_c.compute_roi_traces_f0_and_reorder_cells(order_type)  # this function also coputes the snr / f0s of the cells
print ("DONE...")

computing roi traces for SNR indexing: 100%|█| 1800/1800 [00:04<00:00, 442.65it/

DONE...


In [9]:
#############################################################
############## VISUALIZE ALL CELLS AND TRACES ###############
#############################################################
#
bmi_c.scale=3  
# <--- decrease to see cell traces better
bmi_c.trace_subsample = 10       # Subsample the time series to go faster;

# visualize traces
bmi_c.visualize_traces_snr_order(bmi_c.std_map)

print ("DONE...")

memmap :  (18000, 512, 512)
DONE...


In [10]:
###############################################################
########### SELECT ENSEMBEL CELLS AND VISUALIZE ###############
###############################################################

# save ensemble rois
bmi_c.ensemble1 = [21,27]
bmi_c.ensemble2 = [8,9]
both = np.hstack((bmi_c.ensemble1, bmi_c.ensemble2))
bmi_c.both = both
print ("all cells:", bmi_c.both)

#
bmi_c.show_traces_ids(bmi_c.both)


all cells: [21 27  8  9]


In [11]:
# ######################################################################
# ########### RECOMPUTE TRACES WITH SINGLE FRAME PRECISION #############
# ######################################################################
bmi_c.trace_subsample = 1        # Subsample the time series to go faster;

# visualize traces
#bmi_c.compute_traces2(std_map, both)
bmi_c.compute_traces_ensembles(bmi_c.std_map)

print ("DONE...")

100%|███████████████████████████████████| 18000/18000 [00:04<00:00, 4123.41it/s]


DONE...


In [12]:
#############################################
########### RUN THRSHOLD SETTER #############
#############################################
# 
bmi_c.sample_rate = 30
bmi_c.post_reward_lockout = 10   # reward lockout in seconds
                                 # TODO: in future load/save this to disk so that BMI 
                                 #   - doesn't use differetn lockout than calibration step
bmi_c.balance_ensemble_rewards_flag = False   #this makes sure that both ensembles elicit a similar number of random rewards
bmi_c.rois_smooth_window = 5     # of frames to use to smooth the realtime signal
bmi_c.smooth_diff_function_flag = True    # use a kernel window to smooth current value

# find 30% reward threshold
bmi_c.reward_rate = 0.35

#gr.find_reward_thresholds_low_and_high()
bmi_c.find_reward_thresholds_high()  # this only rewards when sound passes specific level
print ("thresholds: ", bmi_c.high)

############################################## 
bmi_c.plot_rewarded_ensembles()

#
bmi_c.reward_rate_scaling_factor = 0.85

#
bmi_c.high = bmi_c.high*bmi_c.reward_rate_scaling_factor
print ("bmi_c.high: scaled by: ", bmi_c.reward_rate_scaling_factor, ", final value:  ", bmi_c.high)


COMPUTED # of roi traces:  40


100%|█████████████████████████████████| 17995/17995 [00:00<00:00, 340527.24it/s]


low, high:  -2.8685450600237186 1.6184590615135401
nsec recording:  600 max # of random rewards (i.e. every 30sec)  20
 @30% reward:  7
updated rewards #:  7 -2.108644211515838 1.1897161314271512
thresholds:  1.1897161314271512
bmi_c.high: scaled by:  0.85 , final value:   1.0112587117130785


In [13]:
#############################################
########### SAVE DATA #######################
#############################################

#    
save_calibration_data(bmi_c)  


Done...


npyio.py (713): Creating an ndarray from ragged nested sequences (which is a list-or-tuple of lists-or-tuples-or ndarrays with different lengths or shapes) is deprecated. If you meant to do this, you must specify 'dtype=object' when creating the ndarray.


[[0.00000000e+00 1.67360326e+09 3.99000000e+02 1.67360326e+09
  1.00000000e+00]
 [5.19000000e+02 1.67360326e+09 9.99000000e+02 1.67360326e+09
  1.00000000e+00]
 [1.09200000e+03 1.67360326e+09 1.21100000e+03 1.67360326e+09
  1.00000000e+00]
 ...
 [           nan            nan            nan            nan
             nan]
 [           nan            nan            nan            nan
             nan]
 [           nan            nan            nan            nan
             nan]]

0 0.0 399.0

1 519.0 999.0

2 1092.0 1211.0

3 1328.0 1699.0

4 1814.0 2715.0

5 3015.0 3500


In [25]:
cell_id =1
plt.figure()
temp = bmi_c.roi_traces[cell_id]
plt.plot((temp - np.median(temp))/np.median(temp))
plt.plot((temp - bmi_c.roi_f0s[cell_id])/bmi_c.roi_f0s[cell_id])
plt.show()

In [26]:
plt.figure()
for c in range(len(contours_local)):
    for k in range(len(contours_local[c])-1):

        plt.plot([contours_local[c][k][0],
                            contours_local[c][k+1][0]],
                           [contours_local[c][k][1],
                            contours_local[c][k+1][1]],
                            c='blue',
                            linewidth=5)
plt.show()


In [16]:
data1 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_regular_filter.npz',
                allow_pickle=True)
e1 = data1['ensemble_activity']
print (e1.shape)


data2 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_5timesteps.npz',
                allow_pickle=True)
e2 = data2['ensemble_activity']
print (e2.shape)

data3 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results_no_filter.npz',
                allow_pickle=True)
e3 = data3['ensemble_activity']
print (e3.shape)

data4 = np.load('/media/cat/4TB1/donato/DON-010800/22-08-15/data/results.npz',
                allow_pickle=True)
e4 = data4['ensemble_activity']
print (e4.shape)


#############################################################
plt.figure()

t = np.arange(e1.shape[1])/30
plt.plot(t,
         e1[0]-e1[1],
         c='blue', 
         linewidth = 4,
         label='current filter')
plt.plot(t,
         e2[0]-e2[1],
         linewidth = 4,
         c='red', label = 'mean last 5 time steps')
plt.plot(t,
         e3[0]-e3[1],
         linewidth = 4,
         c='green', label = 'no filter')
plt.plot(t,
         e4[0]-e4[1],
         linewidth = 4,
         c='black', label = 'graded filter last 5 time steps')


plt.xlabel("time (sec)", fontsize=16)
plt.ylabel("Ensemble 1 - Ensemble 2", fontsize=16)
plt.xticks(fontsize=16)
plt.yticks(fontsize=16)
plt.plot([t[0],t[-1]],
         [0,0],
         '--',
        c='black')
plt.legend()
plt.show()

(2, 3000)
(2, 3000)
(2, 3000)
(2, 3000)


In [3]:
def get_octave_frequencies(low_freq,
						   high_freq,
						   octave_size=0.25):
	#
	octaves = []

	#
	octaves.append(low_freq)
	temp = low_freq
	while True:
		temp = temp * (1 + octave_size)
		if temp > high_freq:
			break
		octaves.append(temp)
	"""
	low_freq = 1000
	high_freq = 16000
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
    
	octaves = 2**(octave_size*x)
    
	#
	return np.array(1000*octaves)
	"""
	octaves = np.arange(int(low_freq/1000), 1+int(high_freq/1000))
	octaves = 2**(octave_size*x)
	return np.array(1000*octaves)

	#

In [4]:
low_freq = 2000
high_freq = 16000

octaves = get_octave_frequencies(low_freq,high_freq,octave_size=0.25)

NameError: name 'x' is not defined